In [33]:
import numpy as np
import pandas as pd

## BLOCK-GENERATOR
#Generates a randomized block of Go/No-Go trials based on the target conflict ratio.
'''
We should specify the block(HC/LC/MC) and the number of trials in that one block(100).
The PC_ratio will tell us how many trials must be congruent
''' 
def create_block(block_name, n_trials, pc_ratio):
    # 1.trial counts depends pc, changing pc to tweak probabilities
    n_pc = int(n_trials*pc_ratio)
    n_pi = n_trials-n_pc
    # Split PC->GW & NAL
    n_gw = n_pc//2
    n_nal = n_pc-n_gw
    # Split PI->NW&GAL
    n_nw = n_pi//2
    n_gal = n_pi-n_nw
    trials = [] # empty list, to which we attach 
    #the keys(block, Pav, trial, cue, action) and values

    ## hint cues and the correct actions
    ##2. PC TRIALS- 
    # GW: cue(+1), action=go(1)
    trials.extend([{'Block': block_name, 'Conflict': 'PC', 'Trial_Type': 'GW', 
                    'Cue_Valence': 1, 'Optimal_Action': 1}] * n_gw)
    # NAL: cue(-1), action=NoGo(0)
    trials.extend([{'Block': block_name, 'Conflict': 'PC', 'Trial_Type': 'NAL', 
                    'Cue_Valence': -1, 'Optimal_Action': 0}] * n_nal)
    
    ##3. PI TRIALS- 
    # NW: cue(+1), action= nogo(0)
    trials.extend([{'Block': block_name, 'Conflict': 'PI', 'Trial_Type': 'NW', 
                     'Cue_Valence': 1, 'Optimal_Action': 0}] * n_nw)
    # GAL:cue (-1), action= go(1)
    trials.extend([{'Block': block_name, 'Conflict': 'PI', 'Trial_Type': 'GAL', 
                    'Cue_Valence': -1, 'Optimal_Action': 1}] * n_gal)
    
    # 4. Shuffle the trials so they appear in a random order within the block
    df = pd.DataFrame(trials)
    df = df.sample(frac=1).reset_index(drop=True)
    return df

In [61]:
## FULL-BLOCKSET GENERATOR | experiment_df
np.random.seed(42) # For reproducibility
# Set how many trials you want per block (e.g., 40 trials * 4 blocks = 160 total trials)
TRIALS_PER_BLOCK = 1000 # 400 trials per person

# Generate the 4 specific blocks
b1 = create_block('B1_MC', TRIALS_PER_BLOCK, pc_ratio=0.50)
b2 = create_block('B2_HC', TRIALS_PER_BLOCK, pc_ratio=0.30)
b3 = create_block('B4_LC', TRIALS_PER_BLOCK, pc_ratio=0.70)

# Combine them sequentially
experiment_df = pd.concat([b1, b2, b3], ignore_index=True)

# Add Trial Numbers and Placeholders for the Agent
experiment_df.insert(0, 'Trial', experiment_df.index + 1)
experiment_df['Agent_Choice'] = np.nan # fill with nan
experiment_df['Agent_Reward'] = np.nan

# Checking output
print("__Checking Output__")
# Count the PC vs PI trials in each block to prove the math is correct
# Groups rows by both block and how much PC or PI in it
summary = experiment_df.groupby(['Block', 'Conflict']).size().unstack(fill_value=0)

# PC percent for each 
summary['% PC'] = (summary['PC'] / (summary['PC'] + summary['PI'])) * 100
print(summary)

print("\n __First 10 Trials of Block 1__")
print(experiment_df.head(5))

# Save the environment without simulating decision NaN for those spaces
experiment_df.to_csv('blank_environment.csv', index=False)

__Checking Output__
Conflict   PC   PI  % PC
Block                   
B1_MC     500  500  50.0
B2_HC     300  700  30.0
B4_LC     700  300  70.0

 __First 10 Trials of Block 1__
   Trial  Block Conflict Trial_Type  Cue_Valence  Optimal_Action  \
0      1  B1_MC       PI         NW            1               0   
1      2  B1_MC       PI         NW            1               0   
2      3  B1_MC       PI         NW            1               0   
3      4  B1_MC       PI         NW            1               0   
4      5  B1_MC       PC        NAL           -1               0   

   Agent_Choice  Agent_Reward  
0           NaN           NaN  
1           NaN           NaN  
2           NaN           NaN  
3           NaN           NaN  
4           NaN           NaN  


## Agent's Simulation Algorithm

In [62]:
from scipy.special import expit as inv_logit

params = {
    'xi': 0.1, #noise rate
    'ep': 0.15, # learning rate
    'b':  0.5, # go bias
    'pi': 1.2, # pavlovian bias
    'rho': 2.0 # reward sensitivity
    # outcome = the rew received this trial (+1,0,1)
    # sv = cue's Q-value for each stimulus/cue.
    # NW having high sv => this cue is highly rewarding
    # q_go, q_ng, sv/q_sv
    # w_go, w_ng
}

In [63]:
def decide(params, qv_g, qv_ng, sv):
    # sv is the stimulus value of the cue itself.
    # Cue/Cue_Valence: +1(rewarding) & -1(aversive)
    
    b=params['b'] # go bias
    pi=params['pi'] # pav bias
    wv_g  = qv_g + b + pi * sv # wt for Go
    wv_ng = qv_ng  # wt for nogo

    ## Probability of Go
    '''
    First of all, output of softmax is between [0.5,1).
    Now:
    Softmax (wt_go - wt_nogo) = pGo
    then:
    pGo becomes (pGo x 1-noise) + 1/2(noise)
    
    This helps in forcing the value of pGo between 0 and 1
    When noise is 1, 
        softmaxed_pGo becomes (pGo*1-1)+1/2(noise) = 0.5
        pGo = 0.5 (perfectly random)
    When noise is 0,
        soft_pGo becomes (pGo*1-0)+1/2(noise) = pGo+0 ; but pGo after softmax is <1
        so 
        pGo ~ 0.9 (due to softmax)
        
    ''' 
    pGo = inv_logit(wv_g - wv_ng)# softmax Go
    # Noise: multiply first, then add — these aren't interchangeable
    pGo = pGo*(1-params['xi'])
    pGo = pGo+(params['xi']/2)
    return pGo

## Testing the pGo according to our table of Q-values and sv
x=float(decide(params, qv_g=0, qv_ng= 0.8, sv=1))
print(f"the probability for Go, given the Q-values for nogo and stimulus type being +1")
print(f"which is a NW trial, we can see the probability of it choosing go is {x:.2f}")

'''
We set the params:
bias(b) = 1.2 and sv(cue for rew) = 0.9

For the cue for rew, bias is high, means 
more tendency to go 

high value of qv_ng implies nogo is correct. 

the agent will keep making error on conflict trials.
    Because the go bias is .5 and pav bias is 1.2 
'''

the probability for Go, given the Q-values for nogo and stimulus type being +1
which is a NW trial, we can see the probability of it choosing go is 0.69


'\nWe set the params:\nbias(b) = 1.2 and sv(cue for rew) = 0.9\n\nFor the cue for rew, bias is high, means \nmore tendency to go \n\nhigh value of qv_ng implies nogo is correct. \n\nthe agent will keep making error on conflict trials.\n    Because the go bias is .5 and pav bias is 1.2 \n'

In [64]:
def update (params, sv, qv_g, qv_ng, outcome, pressed):
    ## Rescorla-Wagner Update
    '''
    Rescorla–Wagner / delta-rule learning equation:
    New value=Old value+α(ρr−Old value)
    new value (stim : NW, NAL, GW, GAL) = alpha*(rho*outcome - old value)
    
    '''
    sv = sv + params['ep']*(params['rho']*outcome - sv)
    #Q-value, Q_go or Q_nogo to update and by how much
    if (pressed):
        qv_g = qv_g + params['ep']*(params['rho']*outcome - qv_g)
    else:
        qv_ng = qv_ng + params['ep']*(params['rho']*outcome - qv_ng)
    return qv_g, qv_ng, sv

In [65]:

    # It pulls towards Go via pi*sv. 
    #=> a combined effect of pavlovian bias and stimulus value
    # when the cue is avoid, it pulls towards Go via pi*sv.
    # qv = Q value of doing go on this cue
    # qv_ng = Q value of doing nogo on this cue.

def run_agent(experiment_df):
    # All the Q-values persist across the BLOCKSET(4)
    # No block-level reset.
    cues=['GW','NAL','NW','GAL']
    qv_g  = {'GW': 0.0, 'NAL': 0.0, 'NW': 0.0, 'GAL': 0.0}
    qv_ng = {'GW': 0.0, 'NAL': 0.0, 'NW': 0.0, 'GAL': 0.0}
    sv    = {'GW': 0.0, 'NAL': 0.0, 'NW': 0.0, 'GAL': 0.0}
    for idx, row in experiment_df.iterrows(): # row index and row as a series
        cue = row['Trial_Type']
        '''
        Decide the pGo based on the Q_values of go, nogo and sv
            for GW, NW, NAL, GAL
            Referencing the Q-table for each stimulus.
        '''
        pGo=decide(params, qv_g[cue], qv_ng[cue], sv[cue])

        '''
        So.. pGo is an RV between pGo and 1, 
        if it falls between:
            0 and pGo -> Go (pGo)
            pGo and 1 -> nogo (1-pGo)
        '''
        action=np.random.binomial(1,pGo) 
        # otherwise, since pGo> pNogo always chooses go
        # instead of choosing pGo that % of the time.

        
        experiment_df.at[idx, 'Agent_Choice'] = action
        correct = (action == row['Optimal_Action'])
        if row['Cue_Valence'] == 1:
            reward = 1 if correct else 0
        else:
            reward = 0 if correct else -1
        experiment_df.at[idx, 'Agent_Reward'] = reward
        ## replaced outcome w rew and pressed w action
        qv_g[cue], qv_ng[cue], sv[cue] = update(params, sv[cue], qv_g[cue], qv_ng[cue], reward, action)
        

for idx, row in experiment_df.iterrows():
    print(f"index: {idx}, \n\nrow: \n{row}")

index: 0, 

row: 
Trial                 1
Block             B1_MC
Conflict             PI
Trial_Type           NW
Cue_Valence           1
Optimal_Action        0
Agent_Choice        NaN
Agent_Reward        NaN
Name: 0, dtype: object
index: 1, 

row: 
Trial                 2
Block             B1_MC
Conflict             PI
Trial_Type           NW
Cue_Valence           1
Optimal_Action        0
Agent_Choice        NaN
Agent_Reward        NaN
Name: 1, dtype: object
index: 2, 

row: 
Trial                 3
Block             B1_MC
Conflict             PI
Trial_Type           NW
Cue_Valence           1
Optimal_Action        0
Agent_Choice        NaN
Agent_Reward        NaN
Name: 2, dtype: object
index: 3, 

row: 
Trial                 4
Block             B1_MC
Conflict             PI
Trial_Type           NW
Cue_Valence           1
Optimal_Action        0
Agent_Choice        NaN
Agent_Reward        NaN
Name: 3, dtype: object
index: 4, 

row: 
Trial                 5
Block             B1_MC
Co

In [66]:
run_agent(experiment_df)
print('Structure of our simulated data')
print(experiment_df[['Trial', 'Block', 'Trial_Type', 'Agent_Choice', 'Agent_Reward']].head(5))

Structure of our simulated data
   Trial  Block Trial_Type  Agent_Choice  Agent_Reward
0      1  B1_MC         NW           1.0           0.0
1      2  B1_MC         NW           1.0           0.0
2      3  B1_MC         NW           1.0           0.0
3      4  B1_MC         NW           1.0           0.0
4      5  B1_MC        NAL           1.0          -1.0


In [67]:
experiment_df['Correct'] = (experiment_df['Agent_Choice'] == experiment_df['Optimal_Action']).astype(int)

print('Accuracy for each type of trials')
print(experiment_df.groupby('Trial_Type')['Correct'].mean().round(3))

Accuracy for each type of trials
Trial_Type
GAL    0.855
GW     0.949
NAL    0.828
NW     0.540
Name: Correct, dtype: float64


In [79]:
experiment_df.to_csv('simulated_agent_4000T.csv', index=False)

# I have yet to check

In [81]:
"Expected the same 4 cues across every block — persistence logic assumes this."
# Parameter retrieval
from scipy.optimize import minimize
CSV_PATH=('simulated_agent_4000T.csv')
df = pd.read_csv(CSV_PATH)
def compute_nll(params_list, df):
    xi, ep, b, pi, rho = params_list
    nll=0
    cues=['GW','NAL','NW','GAL']
    qv_g  = {'GW': 0.0, 'NAL': 0.0, 'NW': 0.0, 'GAL': 0.0}
    qv_ng = {'GW': 0.0, 'NAL': 0.0, 'NW': 0.0, 'GAL': 0.0}
    sv    = {'GW': 0.0, 'NAL': 0.0, 'NW': 0.0, 'GAL': 0.0}    
    for  idx, row in df.iterrows():
        cue=row['Trial_Type']
        pressed = int(row['Agent_Choice'])    
        
        wv_g  = qv_g[cue] + b + pi * sv[cue]
        wv_ng = qv_ng[cue]
        # convert the diff into a probability using sigmoid
        pGo = inv_logit(wv_g - wv_ng) 
        pGo = pGo*(1-xi) # scale down by noise rate
        pGo = pGo+(xi/2) # add back random noise floor
        if pressed: # means the pressed is yes, or value =1
            likelihood = pGo
        else:
            likelihood = 1 -pGo
        likelihood = np.clip(likelihood, 1e-10, 1 - 1e-10)
        nll-=np.log(likelihood)
        
        outcome = row['Agent_Reward']
        sv[cue]=sv[cue] +ep *(rho*outcome -sv[cue])
        if (pressed):
            qv_g[cue]= qv_g[cue] +ep *(rho*outcome -qv_g[cue])
        else:
            qv_ng[cue]= qv_ng[cue] +ep *(rho*outcome -qv_ng[cue])
    return nll

In [83]:
x0=[0.1,0.2, 0.3, 1.0, 2.0]
bounds = [
    (0.001, 0.99),   # xi
    (0.001, 0.99),   # ep
    (-5, 5),         # b
    (-5, 5),         # pi
    (0.01, 10),      # rho
]

# Checking counts
expected_n = TRIALS_PER_BLOCK * 3
assert len(df) == expected_n, (
    f"Loaded {len(df)} trials but TRIALS_PER_BLOCK={TRIALS_PER_BLOCK} implies "
    f"{expected_n}. df is stale — rerun the simulation cells first."
)
## Computing negative log likelihood

result = minimize(compute_nll, x0, args=(df,), method='L-BFGS-B',bounds=bounds)


In [84]:
print(f"        recovered params: {result.x.round(2)}")
print(f"param values: {params.values()}")
print(f"model params: {params.keys()}")

        recovered params: [0.07 0.11 0.52 1.14 1.99]
param values: dict_values([0.1, 0.15, 0.5, 1.2, 2.0])
model params: dict_keys(['xi', 'ep', 'b', 'pi', 'rho'])
